<a href="https://colab.research.google.com/github/olawoye/nla_phd/blob/main/notebooks/setup_activations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Lets instal basic libraries
!pip install -q transformers
!pip install -q accelerate
!pip install -q datasets
!pip install -q pyarrow
!pip install -q pandas
!pip install -q sentencepiece

# For Gemma
!pip install -q torch

import json
import os
import torch
import pandas as pd

#fix issuess saving to my github
!pip install ipywidgets

# setup our github save behaviour
import glob
from google.colab import userdata
import shutil

import os
import shutil
import glob
import pandas as pd
from google.colab import userdata

# --- 1. CONFIGURATION ---
GH_USER = "olawoye"
REPO_NAME = "nla_phd"
LOCAL_REPO_PATH = f"/content/{REPO_NAME}"
TARGET_LAYER = 8
REPO_DOMAIN = "github.com"

# --- HARD RESET SYSTEM DIRECTORY PATH ---
# Forces the entire notebook engine out of any bad background directory state
os.chdir("/content")

# --- 2. SECURE CLONE OR UPDATE ---
is_valid_git_repo = os.path.exists(LOCAL_REPO_PATH) and os.path.exists(os.path.join(LOCAL_REPO_PATH, ".git"))

if not is_valid_git_repo:
    if os.path.exists(LOCAL_REPO_PATH):
        print("⚠️ Cleaning up partial directory structure...")
        shutil.rmtree(LOCAL_REPO_PATH)

    print(" Fresh Colab session detected. Cloning your repository cleanly...")
    try:
        gh_token = userdata.get('GITHUB_TOKEN')

        # FIXED STRING STRUCTURE: Explicitly split elements to block URL mangling bugs
        domain = REPO_DOMAIN
        repo_url = f"https://{gh_token}@{domain}/{GH_USER}/{REPO_NAME}.git"

        # Run clone using precise destination tracking
        !git clone "{repo_url}" "{LOCAL_REPO_PATH}"

    except Exception as e:
        print(f"❌ Error accessing GitHub token secret: {e}")
        print("Setting up local folders as an offline fallback...")
        os.makedirs(LOCAL_REPO_PATH, exist_ok=True)
else:
    print(" Repository exists and is valid. Pulling latest changes...")
    os.chdir(LOCAL_REPO_PATH)
    !git pull origin main

# Always drop back to system root after git tasks
os.chdir("/content")

# --- 3. ENSURE DIRECTORY STRUCTURE ---
os.makedirs(os.path.join(LOCAL_REPO_PATH, "data"), exist_ok=True)
os.makedirs(os.path.join(LOCAL_REPO_PATH, "notebooks"), exist_ok=True)

# --- 4. SAFE .GITIGNORE CREATION ---
gitignore_content = """
data/large_combinations.parquet
data/*_50k.parquet
.ipynb_checkpoints/
"""
gitignore_path = os.path.join(LOCAL_REPO_PATH, ".gitignore")
with open(gitignore_path, "w") as f:
    f.write(gitignore_content.strip())
print("✓ .gitignore verified and updated.")

# --- 5. AUTOMATED PROGRESS RECOVERY ---
def load_existing_progress():
    data_dir = os.path.join(LOCAL_REPO_PATH, "data")
    parquet_files = glob.glob(os.path.join(data_dir, "checkpoint_*.parquet"))

    if not parquet_files:
        print("\n No previous saved checkpoints found in your repository. Starting fresh (Index: 0).")
        return pd.DataFrame(), 0

    print(f"\n Found {len(parquet_files)} saved checkpoint files. Reloading into memory...")
    df_list = [pd.read_parquet(f) for f in parquet_files]
    combined_df = pd.concat(df_list, ignore_index=True)

    next_index = int(combined_df["sample_index"].max() + 1) if "sample_index" in combined_df.columns else 0
    print(f"✓ Successfully reloaded {len(combined_df)} historical activations.")
    print(f" Reset complete. Ready to run starting at sample index: {next_index}")
    return combined_df, next_index

# Run the recovery function automatically
historical_data, starting_sample_index = load_existing_progress()







 Repository exists and is valid. Pulling latest changes...
From https://github.com/olawoye/nla_phd
 * branch            main       -> FETCH_HEAD
   960d1ad..5d6fbb2  main       -> origin/main
Already up to date.
✓ .gitignore verified and updated.

 Found 2 saved checkpoint files. Reloading into memory...
✓ Successfully reloaded 150 historical activations.
 Reset complete. Ready to run starting at sample index: 100


In [2]:
# Setup my project folder and additional configuration
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = f"/content/drive/MyDrive/{REPO_NAME}"

USE_GOOGLE_DRIVE = False  # Set to True to save to Drive, False to keep local

from google.colab import userdata

# Securely retrieve the token from Colab's secret vault
github_token = userdata.get('GITHUB_TOKEN')

# Inject it into your system environment variables for Git to use
os.environ['GITHUB_TOKEN'] = github_token


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Huggin face approval
from huggingface_hub import login

login()

In [4]:
# So I'm kind of experiementing with either Gemma-2-2B and Qwen2.5-Coder-3B (because its seems with better code inclined)
# but Gemma seems to better SAE ecosytem to leverage faster

# Gemma Option
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "google/gemma-2-2b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

# Should we decide on Qwen
'''
model_name = "Qwen/Qwen2.5-Coder-3B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)
'''

# Lets see
print(model)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

In [5]:
# Lets run a simple inference from the model


if USE_GOOGLE_DRIVE:
    save_dir = PROJECT_DIR + "/colab_outputs"
else:
    save_dir = "/content/local_colab_outputs"

os.makedirs(save_dir, exist_ok=True)

# --- PROMPTS FOR YOUR BULK RUN ---
prompts_list = [
    "Write a Python function that computes factorial.",
    "Write a Python function that checks prime numbers.",
    # Add your thousands of prompts here...
]

# --- manually define the Gemma chat template (our base model doesnt carry this)
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{'<start_of_turn>' + message['role'] + '\n' + message['content'] + '<end_of_turn>\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "{{'<start_of_turn>assistant\n'}}"
        "{% endif %}"
    )

# --- BULK RUN LOOP ---
for idx, prompt_text in enumerate(prompts_list):
    chat = [{"role": "user", "content": f"{prompt_text} Output only the code."}]
    prompt = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Inference
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        output_hidden_states=True,
        return_dict_in_generate=True,
    )


    # --- SHAPE INSPECTION PRINTING BLOCK ---

    # 1. output.hidden_states is a tuple where each element is a generation step
    num_steps = len(output.hidden_states)

    print("Shape:\n")

    # Let's inspect the very first step (Step 0) of the generation
    first_step_layers = output.hidden_states[0]

    # Extract Layer 8 (index 8) from this step
    # This gives you the full tensor before any slicing or memory optimization
    layer_8 = first_step_layers[8]

    # Print the code and dynamic shape dynamically
    print("print(layer_8.shape)\n")
    print(f"{tuple(layer_8.shape)}\n")

    # 2. Extract dimensions from the shape tensor
    batch_size, sequence_length, hidden_dimensions = layer_8.shape

    print("Meaning:\n")
    print(f"batch size = {batch_size}\n")
    print(f"{sequence_length} tokens\n")
    print(f"{hidden_dimensions} hidden dimensions\n")
    print("-" * 40)



    # Text extraction
    input_length = inputs.input_ids.shape[1]
    generated_tokens = output.sequences[0, input_length:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # Layer 8 Extraction (Memory Optimized)
    layer_8_activations = []
    for step in range(len(output.hidden_states)):
        # Grab layer 8, extract last token slice, drop to CPU list immediately
        last_token_act = output.hidden_states[step][8][0, -1, :]
        layer_8_activations.append(last_token_act.detach().cpu().tolist())

    # Build individual save payload
    output_data = {
        "prompt": prompt_text,
        "generated_code": generated_text,
        "layer_8_activations": layer_8_activations,
    }

    # Save incrementally to prevent data loss if Colab disconnects
    file_path = os.path.join(save_dir, f"result_{idx}.json")
    with open(file_path, "w") as f:
        json.dump(output_data, f)

print(f"Bulk run completed! Files saved in: {save_dir}")


Shape:

print(layer_8.shape)

(1, 22, 2304)

Meaning:

batch size = 1

22 tokens

2304 hidden dimensions

----------------------------------------
Shape:

print(layer_8.shape)

(1, 23, 2304)

Meaning:

batch size = 1

23 tokens

2304 hidden dimensions

----------------------------------------
Bulk run completed! Files saved in: /content/local_colab_outputs


In [6]:
# Lets get ready to do this for larger data

NUM_SAMPLES_TO_PROCESS = 100  # I can easily set to 10, 100, 10000, or 50000
CHECKPOINT_EVERY = 50        # Save data periodically to prevent loss
TARGET_LAYER = 8             # The internal transformer layer to extract according to documen
TEXT_TRUNCATION_LIMIT = 1000 # Character limit per document to keep generation fast
OUTPUT_FILENAME = "gemma_activations_final.parquet"

print(f"COnfiguring to to process {NUM_SAMPLES_TO_PROCESS} samples with checkpoints every {CHECKPOINT_EVERY} steps.")

# --- LOAD DATASET ---
from datasets import load_dataset

# Downloading/Loading OpenWebText dataset from Hugging Face (we use streaming=True download all 40GB of OpenWebText to Colab
dataset = load_dataset("Skylion007/openwebtext", split="train", streaming=True)
dataset_iter = iter(dataset)

# Test print the first item sample snippet to see it works
test_item = next(dataset_iter)
print(test_item["text"][:200] + "...")


COnfiguring to to process 100 samples with checkpoints every 50 steps.


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Port-au-Prince, Haiti (CNN) -- Earthquake victims, writhing in pain and grasping at life, watched doctors and nurses walk away from a field hospital Friday night after a Belgian medical team evacuated...


In [7]:
# Put layer activation in a single func to re-use easily and keep tweaks in one plac

def extract_layer_activation(text_input, layer_index=8):
    """
    Passes text through the model and extracts the hidden state activation
    vector of the final token at a specific layer.
    """
    # Truncate text to keep sequence lengths uniform and fast
    truncated_text = text_input[:TEXT_TRUNCATION_LIMIT]

    inputs = tokenizer(
        truncated_text,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    # We use torch.no_grad() because we are NOT training the LLM; this saves massive memory
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Shape: [1, seq_len, 2304]
    full_layer_tensor = outputs.hidden_states[layer_index]

    # Slicing out [0, -1] yields the final token vector -> Shape: [2304]
    final_token_vector = full_layer_tensor[0, -1, :]

    # CRITICAL STEP FOR 50K RUNS: Move to CPU and cast to numpy to free up GPU cache
    activation_array = final_token_vector.detach().cpu().numpy()

    return activation_array, truncated_text


In [8]:
# and so we start...

# Calculate the target stop index based on previous history
target_stop_index = starting_sample_index + NUM_SAMPLES_TO_PROCESS

print(f"Resuming run from index {starting_sample_index}.")
print(f"Will extract {NUM_SAMPLES_TO_PROCESS} samples. Stop target index is: {target_stop_index}")

# Initialize a fresh local list for this session's progress
records = []

# --- THE ADJUSTED LOOP ---
for i in range(target_stop_index):
    try:
        # Fetch the text document from the OpenWebText stream
        document = next(dataset_iter)

        # INCREMENTAL FIX: Skip data you have already extracted in past daily sessions
        if i < starting_sample_index:
            continue

        raw_text = document["text"]

        # Run inference and capture layer 8 activation vector
        activation, processed_text = extract_layer_activation(raw_text, layer_index=TARGET_LAYER)

        # Append record using the unique, non-overlapping index 'i'
        records.append({
            "sample_index": i,
            "text": processed_text,
            "layer": TARGET_LAYER,
            "activation": activation.tolist()
        })

    except StopIteration:
        print("Reached the end of the dataset stream.")
        break
    except Exception as e:
        print(f"Skipped index {i} due to an error: {e}")
        continue

    # Checkpoint saving rules remain the same
    if (i + 1) % CHECKPOINT_EVERY == 0:
        checkpoint_df = pd.DataFrame(records)
        # Use the precise dynamic index in the file name to prevent over-writing previous files
        checkpoint_path = os.path.join(LOCAL_REPO_PATH, f"data/checkpoint_step_{i + 1}.parquet")
        checkpoint_df.to_parquet(checkpoint_path)
        print(f"Saved session checkpoint: {checkpoint_path} (Records this session: {len(records)})")

print(f"\nSession processing completed. Added {len(records)} new activations.")


Resuming run from index 100.
Will extract 100 samples. Stop target index is: 200
Saved session checkpoint: /content/nla_phd/data/checkpoint_step_150.parquet (Records this session: 50)
Saved session checkpoint: /content/nla_phd/data/checkpoint_step_200.parquet (Records this session: 100)

Session processing completed. Added 100 new activations.


In [9]:
# Add everything into a final DataFrame
final_df = pd.DataFrame(records)

# Save to file
final_df.to_parquet(OUTPUT_FILENAME)
print(f"final dataset successfully created: {OUTPUT_FILENAME}")

# Lets verify
print("\n--- DATASET VERIFICATION ---")
print(f"Total Rows: {final_df.shape[0]}")
print(f"Columns: {list(final_df.columns)}")

# Check the vector length of the first row's activation column
vector_length = len(final_df["activation"].iloc[0])
print(f"Extracted Vector Hidden Dimension Size: {vector_length} (Expected: 2304)")

# Show final layout preview
final_df.head(2)


final dataset successfully created: gemma_activations_final.parquet

--- DATASET VERIFICATION ---
Total Rows: 100
Columns: ['sample_index', 'text', 'layer', 'activation']
Extracted Vector Hidden Dimension Size: 2304 (Expected: 2304)


,sample_index,text,layer,activation
0,100,People travelling to the Czech Republic have b...,8,"[0.58254474401474, 0.6995223760604858, -0.2937..."
1,101,DigitalGlobe is enlisting the crowd to scan an...,8,"[0.46592938899993896, -1.0448801517486572, -1...."


In [10]:
# Save Progress to Hithub


GH_TOKEN = userdata.get('GITHUB_TOKEN')
GH_USER = "olawoye"

gh_token = userdata.get('GITHUB_TOKEN')
domain = "github.com"
repo_url = f"https://{gh_token}@{domain}/{GH_USER}/{REPO_NAME}.git"

# Configure Git Identity
!git config --global user.email "ifeoluwa.olawoye@gmail.com"
!git config --global user.name "Ifeoluwa Olawoye"

# Pull latest remote changes
%cd /content/{REPO_NAME}
!git pull origin main

# Add files, commit, and push (we exclude fiant files)
!git add .
!git commit -m "NLA project auto-sync checkpoints and updates"
!git push {repo_url} main

print("\n nla project progress synced with GitHub.")


/content/nla_phd
From https://github.com/olawoye/nla_phd
 * branch            main       -> FETCH_HEAD
Already up to date.
[main d2d07db] NLA project auto-sync checkpoints and updates
 2 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 data/checkpoint_step_150.parquet
 create mode 100644 data/checkpoint_step_200.parquet
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.22 MiB | 3.12 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/olawoye/nla_phd.git
   5d6fbb2..d2d07db  main -> main

 nla project progress synced with GitHub.
